In [2]:
# Let's calculate the correlation of continuation metrics between tacotron2 and vits.
# We have PPL, VERT, and LLM-as-a-Judge score.

In [2]:
# First, load settings with the same N-K-temperature

tacotron_settings = set()
vits_settings = set()
with open("csv/continuation_result_10s.csv") as f:
    for line in f:
        setting, _, _, _, temperature, _ = line.split(",", 5)
        if "tacotron2" not in setting and "vits" not in setting:
            continue
        if not temperature:
            continue
        NK = setting.split("-", 1)[1]
        if "tacotron2" in setting:
            tacotron_settings.add(f"{NK}-{temperature}")
        elif "vits" in setting:
            vits_settings.add(f"{NK}-{temperature}")
common_settings = tacotron_settings & vits_settings
print("tacotron2 settings:", len(tacotron_settings))
print("vits settings:", len(vits_settings))
print("common settings:", len(common_settings))
# These settings shares same unit sequence
common_settings = sorted(list(common_settings))
print(common_settings)

tacotron2 settings: 21
vits settings: 20
common settings: 11
['20-128-0.6', '20-2048-0.6', '20-256-0.6', '20-4096-0.6', '20-512-0.6', '40-1024-0.7', '40-256-0.6', '40-4096-0.7', '80-4096-0.5', '80-512-0.7', '80-8192-0.7']


In [3]:
# Calculate correlation of ppl scores for common settings
from scipy.stats import spearmanr

tacotron2_ppls = {}
vits_ppls = {}
for setting in common_settings:
    with open(f"csv/ppl_10s/tacotron2/{setting}.txt") as f, open(f"csv/ppl_10s/vits/{setting}.txt") as g:
        for line_t, line_v in zip(f, g):
            wav_id_t, ppl_t = line_t.strip().split("|", 1)
            wav_id_v, ppl_v = line_v.strip().split("|", 1)
            if wav_id_t == "PPL_corpus":
                continue
            assert wav_id_t == wav_id_v
            tacotron2_ppls[f"{setting}-{wav_id_t}"] = float(ppl_t)
            vits_ppls[f"{setting}-{wav_id_v}"] = float(ppl_v)

print(len(tacotron2_ppls), len(vits_ppls))
# 2274 2274
correlation, p_value = spearmanr(list(tacotron2_ppls.values()), list(vits_ppls.values()))
print(f"Spearman correlation coefficient: {correlation:.4f} (p-value: {p_value:.4e})")
# Spearman correlation coefficient: 0.7569 (p-value: 0.0000e+00)

2274 2274
Spearman correlation coefficient: 0.7569 (p-value: 0.0000e+00)


In [4]:
# Calculate correlation of vert scores for common settings
from asr_eval_bleu import all_scores
from scipy.stats import spearmanr

tacotron2_verts = {}
vits_verts = {}
for setting in common_settings:
    all_scores_t = all_scores(f"transcription_cut_10s/tacotron2/{setting}.txt")
    all_scores_v = all_scores(f"transcription_cut_10s/vits/{setting}.txt")
    t_verts = {f'{setting}-{wav_id}': m for wav_id, m in all_scores_t["VERT"]}
    v_verts = {f'{setting}-{wav_id}': m for wav_id, m in all_scores_v["VERT"]}
    tacotron2_verts.update(t_verts)
    vits_verts.update(v_verts)

print(len(tacotron2_verts), len(vits_verts))
common_keys = set(tacotron2_verts.keys()) & set(vits_verts.keys())
tacotron2_vals = [tacotron2_verts[k] for k in common_keys]
vits_vals = [vits_verts[k] for k in common_keys]
correlation, p_value = spearmanr(tacotron2_vals, vits_vals)
print(f"Spearman correlation coefficient: {correlation:.4f} (p-value: {p_value:.4e})")
# it takes 2m 38.2s to run, and the result is:
# Spearman correlation coefficient: 0.8040 (p-value: 0.0000e+00)

2274 2274
Spearman correlation coefficient: 0.8040 (p-value: 0.0000e+00)


In [5]:
# Calculate correlation of llm-as-a-judge scores for common settings

from collections import defaultdict
import itertools
from pathlib import Path
import numpy as np
from scipy.stats import spearmanr

def id_to_score_dict(lines):
    score_dict = {}
    for line in lines[1:]: # skip header
        wav_id, score, _ = line.strip().split(",", 2)
        score_dict[wav_id] = float(score)
    return score_dict

tacotron2_llmscores = defaultdict(list)
vits_llmscores = defaultdict(list)
score_dict = defaultdict(lambda: defaultdict(list))
for setting_X, setting_Y in itertools.product(common_settings, repeat=2):
    # {model}-{N}-{K}-{temperature}_vs_{model}-{N}-{K}-{temperature}
    tacotron2_path = Path("pairwise_10s") / f"tacotron2-{setting_X}_vs_tacotron2-{setting_Y}"
    vits_path = Path("pairwise_10s") / f"vits-{setting_X}_vs_vits-{setting_Y}"
    with open(tacotron2_path / "summary.txt") as f, open(vits_path / "summary.txt") as g:
        tacotron2_lines = f.readlines()
        vits_lines = g.readlines()
    tacotron2_score_dict = id_to_score_dict(tacotron2_lines)
    vits_score_dict = id_to_score_dict(vits_lines)

    # Ensure both dictionaries have the same keys (common wav_ids)
    # This is required because in LLM sometimes fails to score some samples, resulting in missing wav_ids in the summary.txt
    common_wav_ids = set(tacotron2_score_dict.keys()) & set(vits_score_dict.keys())
    # print(len(common_wav_ids), "common wav ids for", setting_X, setting_Y)
    for wav_id in common_wav_ids:
        tacotron2_llmscores[f"{setting_X}-{wav_id}"].append(tacotron2_score_dict[wav_id])
        vits_llmscores[f"{setting_X}-{wav_id}"].append(vits_score_dict[wav_id])

print(len(tacotron2_llmscores), len(vits_llmscores))
tacotron2_llmscores_flatten = [score for scores in tacotron2_llmscores.values() for score in scores]
vits_llmscores_flatten = [score for scores in vits_llmscores.values() for score in scores]
correlation, p_value = spearmanr(tacotron2_llmscores_flatten, vits_llmscores_flatten)
print(f"Spearman correlation coefficient (sample-wise): {correlation:.4f} (p-value: {p_value:.4e})")

tacotron2_llmscores_means = {k: np.mean(v) for k, v in tacotron2_llmscores.items()}
vits_llmscores_means = {k: np.mean(v) for k, v in vits_llmscores.items()}
correlation, p_value = spearmanr(list(tacotron2_llmscores_means.values()), list(vits_llmscores_means.values()))
print(f"Spearman correlation coefficient (average): {correlation:.4f} (p-value: {p_value:.4e})")

2274 2274
Spearman correlation coefficient (sample-wise): 0.4068 (p-value: 0.0000e+00)
Spearman correlation coefficient (average): 0.6014 (p-value: 8.8167e-224)


In [6]:
# MMOS mapping from {setting}-{wav_id} to scores
mmos_scores = defaultdict(list)
for keys in tacotron2_ppls.keys():
    setting, wav_id1, wav_id2 = keys.rsplit("-", 2)
    wav_id = f"{wav_id1}-{wav_id2}"
    with open(f"mmos_results/summary/{setting}.csv") as f:
        for line in f.readlines()[1:]:  # skip header
            # 5cae6a77b38ea60016e54889,LJ049-0061-0,3
            rater_id, sample_id, score = line.strip().split(",")
            if sample_id == wav_id:
                mmos_scores[f"{setting}-{wav_id}"].append(float(score))

mmos_scores_mean = {k: np.mean(v) for k, v in mmos_scores.items()}
print(len(mmos_scores_mean))


550


In [ ]:
# take correlations between different continuation metrics (all tacotron2)
# print(list(tacotron2_ppls.keys())[:10])
# print(list(vits_ppls.keys())[:10])
# print(list(tacotron2_verts.keys())[:10])
# print(list(vits_verts.keys())[:10])
# print(list(tacotron2_llmscores_means.keys())[:10])
# print(list(vits_llmscores_means.keys())[:10])
assert set(tacotron2_ppls.keys()) == set(vits_ppls.keys()) == set(tacotron2_verts.keys()) == set(vits_verts.keys()) == set(tacotron2_llmscores_means.keys()) == set(vits_llmscores_means.keys())
print("keys are the same for all metrics, ready for correlation analysis")

mmos_values = []
ppl_values = []
vert_values = []
llmscore_values = []
for key in mmos_scores_mean.keys():
    mmos_values.append(mmos_scores_mean[key])
    llmscore_values.append(tacotron2_llmscores_means[key])
    ppl_values.append(tacotron2_ppls[key])
    vert_values.append(tacotron2_verts[key])

print(len(mmos_values), len(llmscore_values), len(ppl_values), len(vert_values))
metrics_pairs = {
    "MMOS vs LLM": (mmos_values, llmscore_values),
    "MMOS vs PPL": (mmos_values, ppl_values),
    "MMOS vs VERT": (mmos_values, vert_values),
    "LLM vs PPL": (ppl_values, llmscore_values),
    "LLM vs VERT": (vert_values, llmscore_values),
}

for pair_name, (x_values, y_values) in metrics_pairs.items():
    correlation, p_value = spearmanr(x_values, y_values)
    print(f"SRCC ({pair_name}): {correlation:.4f} (p-value: {p_value:.6e})")


keys are the same for all metrics, ready for correlation analysis
SRCC (MMOS vs LLM): 0.3232 (p-value: 7.657537e-15)
SRCC (MMOS vs PPL): -0.1052 (p-value: 1.359873e-02)
SRCC (MMOS vs VERT): 0.0309 (p-value: 4.702042e-01)
SRCC (LLM vs PPL): -0.3635 (p-value: 1.277624e-18)
SRCC (LLM vs VERT): -0.0597 (p-value: 1.620579e-01)
